# 數據準備與模型選擇

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明資料品質為何會影響 AI 模型表現。
2. 使用 pandas 檢查缺失值、重複值與異常值。
3. 對數值、類別、時間與文字欄位進行基礎特徵工程。
4. 使用特徵選擇與降維概念觀察資料結構。
5. 根據問題類型與模型表現，選擇合適的機器學習模型。

本練習以「客戶是否會流失」作為範例情境，模擬企業從內部 CRM、交易紀錄與使用行為資料中準備模型訓練資料。


In [ ]:
# ── 環境設定與範例資料建立 ─────────────────────────────
# 載入本章節所需的 Python 套件，並建立一份包含缺失值、重複值、異常值、類別欄位、時間欄位與文字欄位的範例資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

np.random.seed(42)

n = 120
customer_ids = [f"C{i:04d}" for i in range(n)]
usage_minutes = np.random.normal(180, 45, n).round(1)
monthly_fee = np.random.normal(850, 180, n).round(0)
support_tickets = np.random.poisson(1.5, n)
plan = np.random.choice(["Basic", "Pro", "Enterprise"], size=n, p=[0.5, 0.35, 0.15])
signup_date = pd.date_range("2025-01-01", periods=n, freq="3D")
comments = np.random.choice([
    "服務穩定 功能足夠",
    "價格偏高 希望改善",
    "客服回覆很快",
    "系統偶爾延遲",
    "介面好用 效率提升"
], size=n)

churn = ((usage_minutes < 150) | (support_tickets >= 3) | (monthly_fee > 1000)).astype(int)

raw_df = pd.DataFrame({
    "customer_id": customer_ids,
    "usage_minutes": usage_minutes,
    "monthly_fee": monthly_fee,
    "support_tickets": support_tickets,
    "plan": plan,
    "signup_date": signup_date,
    "comment": comments,
    "churn": churn
})

raw_df.loc[[5, 18, 32], "usage_minutes"] = np.nan
raw_df.loc[[11, 47], "plan"] = np.nan
raw_df.loc[7, "monthly_fee"] = 5000
raw_df = pd.concat([raw_df, raw_df.iloc[[3]]], ignore_index=True)

print("資料筆數與欄位數：", raw_df.shape)
print("\n前 5 筆資料：")
print(raw_df.head())
print("\n缺失值統計：")
print(raw_df.isna().sum())


## 核心概念說明

在 AI 專案中，資料準備通常比模型訓練更耗時，也更影響最終成果。若資料有缺失、重複、異常或邏輯不一致，即使使用複雜模型，也可能產生不可靠的預測結果。

常見資料品質檢核包含：

- 完整性：是否缺少重要欄位或欄位值。
- 一致性：欄位之間是否符合業務邏輯，例如年齡不可為負數。
- 準確性：資料是否能反映真實情況。
- 即時性：資料是否足夠新，能反映目前狀態。
- 唯一性：主鍵資料是否重複，例如顧客編號或交易編號。

在實務上，資料清理通常會先處理缺失值、重複值與異常值，再進行特徵工程與模型選擇。


In [ ]:
# ── 示範：資料清理與品質檢核 ────────────────────────────
# 這段程式碼示範如何使用 pandas 檢查並處理缺失值、重複值與異常值，對應教材中的資料清理與品質檢核流程。

import numpy as np
import pandas as pd

np.random.seed(42)
n = 120
raw_df = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(n)],
    "usage_minutes": np.random.normal(180, 45, n).round(1),
    "monthly_fee": np.random.normal(850, 180, n).round(0),
    "support_tickets": np.random.poisson(1.5, n),
    "plan": np.random.choice(["Basic", "Pro", "Enterprise"], size=n, p=[0.5, 0.35, 0.15]),
    "signup_date": pd.date_range("2025-01-01", periods=n, freq="3D"),
    "churn": 0
})
raw_df["churn"] = ((raw_df["usage_minutes"] < 150) | (raw_df["support_tickets"] >= 3) | (raw_df["monthly_fee"] > 1000)).astype(int)
raw_df.loc[[5, 18, 32], "usage_minutes"] = np.nan
raw_df.loc[[11, 47], "plan"] = np.nan
raw_df.loc[7, "monthly_fee"] = 5000
raw_df = pd.concat([raw_df, raw_df.iloc[[3]]], ignore_index=True)

print("清理前資料筆數：", len(raw_df))
print("清理前缺失值：")
print(raw_df.isna().sum())

clean_df = raw_df.copy()
clean_df = clean_df.drop_duplicates(subset="customer_id")
clean_df["usage_minutes"] = clean_df["usage_minutes"].fillna(clean_df["usage_minutes"].median())
clean_df["plan"] = clean_df["plan"].fillna(clean_df["plan"].mode()[0])

q1 = clean_df["monthly_fee"].quantile(0.25)
q3 = clean_df["monthly_fee"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
clean_df["monthly_fee_is_outlier"] = clean_df["monthly_fee"] > upper_bound
clean_df.loc[clean_df["monthly_fee"] > upper_bound, "monthly_fee"] = clean_df["monthly_fee"].median()

quality_report = pd.DataFrame({
    "指標": ["完整性", "唯一性", "異常值處理"],
    "檢查結果": [
        f"剩餘缺失值總數：{clean_df.isna().sum().sum()}",
        f"顧客編號重複數：{clean_df.duplicated(subset='customer_id').sum()}",
        f"偵測並處理月費異常值數：{clean_df['monthly_fee_is_outlier'].sum()}"
    ]
})

print("\n清理後資料筆數：", len(clean_df))
print("\n資料品質檢核報告：")
print(quality_report)


## 特徵工程說明

特徵工程是將原始資料轉換成模型可學習格式的過程。良好的特徵工程可以讓模型更容易找出資料中的規律。

常見處理方式包含：

- 數值特徵轉換：例如標準化，讓不同單位的數值欄位能在相近尺度上比較。
- 類別特徵處理：例如 One-hot Encoding，將無序類別轉成多個 0/1 欄位。
- 時間特徵擴增：從日期中取出月份、星期幾、是否週末等資訊。
- 文字特徵轉換：使用 TF-IDF 將文字轉成數值特徵，作為輕量版的文字向量化示範。
- 特徵選擇與降維：挑選有用欄位，或使用 PCA 將高維資料壓縮到較低維度。


In [ ]:
# ── 示範：數值、類別、時間與文字特徵工程 ──────────────────────
# 這段程式碼示範標準化、One-hot Encoding、時間欄位拆解與 TF-IDF 文字特徵，對應教材中的特徵處理流程。

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

np.random.seed(7)
df = pd.DataFrame({
    "usage_minutes": [120, 220, 180, 95, 260, 140],
    "monthly_fee": [799, 1299, 999, 699, 1599, 899],
    "support_tickets": [3, 0, 1, 4, 0, 2],
    "plan": ["Basic", "Enterprise", "Pro", "Basic", "Enterprise", "Pro"],
    "signup_date": pd.to_datetime(["2025-01-03", "2025-01-05", "2025-02-10", "2025-02-16", "2025-03-21", "2025-03-29"]),
    "comment": [
        "價格偏高 希望改善",
        "服務穩定 功能足夠",
        "客服回覆很快",
        "系統偶爾延遲",
        "介面好用 效率提升",
        "希望增加更多功能"
    ]
})

numeric_features = ["usage_minutes", "monthly_fee", "support_tickets"]
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(df[numeric_features])
numeric_scaled_df = pd.DataFrame(
    numeric_scaled,
    columns=[f"{col}_scaled" for col in numeric_features]
)

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
plan_encoded = encoder.fit_transform(df[["plan"]])
plan_encoded_df = pd.DataFrame(
    plan_encoded,
    columns=encoder.get_feature_names_out(["plan"])
)

df["signup_month"] = df["signup_date"].dt.month
df["signup_weekday"] = df["signup_date"].dt.weekday
df["signup_is_weekend"] = df["signup_weekday"].isin([5, 6]).astype(int)
time_features_df = df[["signup_month", "signup_weekday", "signup_is_weekend"]]

tfidf = TfidfVectorizer(max_features=6)
text_features = tfidf.fit_transform(df["comment"]).toarray()
text_features_df = pd.DataFrame(
    text_features,
    columns=[f"tfidf_{term}" for term in tfidf.get_feature_names_out()]
)

feature_df = pd.concat(
    [numeric_scaled_df, plan_encoded_df, time_features_df, text_features_df],
    axis=1
)

pca = PCA(n_components=2, random_state=7)
pca_result = pca.fit_transform(feature_df)
pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"])

print("特徵工程後欄位數：", feature_df.shape[1])
print("\n特徵資料前 3 筆：")
print(feature_df.head(3))
print("\nPCA 降維結果：")
print(pca_df.round(3))
print("\nPCA 解釋變異比例：", np.round(pca.explained_variance_ratio_, 3))
